In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 测试torch版本和cuda是否安装成功
print(torch.__version__)
print(torch.cuda.is_available())

# 测试torch的功能
x = torch.rand(5, 3)
print(x)

2.6.0+cpu
False
tensor([[0.9159, 0.0731, 0.1000],
        [0.9715, 0.8297, 0.8398],
        [0.2511, 0.7606, 0.7495],
        [0.0295, 0.8762, 0.9805],
        [0.8414, 0.4512, 0.0469]])


### 数据预处理部分

In [10]:
import pandas as pd

movies = pd.read_csv('./movies.dat', sep='::', engine='python', names=['movie_id', 'title', 'genres'], encoding='latin-1')
ratings = pd.read_csv('./ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])
users = pd.read_csv('./users.dat', sep='::', engine='python', names=['user_id', 'gender', 'age', 'occupation', 'zip'])

In [11]:
movies

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanj(1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [12]:
ratings

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [13]:
positive_samples = ratings[['user_id', 'movie_id', 'rating']]

In [14]:
# 计算每个物品被打分的次数
item_popularity = ratings.groupby('movie_id').size().reset_index(name='count')

# 计算打分次数的0.75次方，就是热门物品被抽到的概率，越热门的物品被抽到的概率越高
item_popularity['weight'] = item_popularity['count'] ** 0.75

In [15]:
item_popularity

,movie_id,count,weight
0,1,2077,307.664489
1,2,701,136.234943
2,3,478,102.228249
3,4,170,47.080026
4,5,296,71.362291
...,...,...,...
3701,3948,862,159.085446
3702,3949,304,72.803991
3703,3950,54,19.920275
3704,3951,40,15.905415


In [16]:
# 获取所有用户和物品的列表
all_users = ratings['user_id'].unique()
all_items = ratings['movie_id'].unique()

# 生成用户-物品的交互矩阵（表示用户是否打过分）
interaction_matrix = ratings.groupby(['user_id', 'movie_id']).size().unstack(fill_value=0)

In [17]:
interaction_matrix

movie_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6036,0,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6037,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6038,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 负样本列表
negative_samples = []

# 对每个用户抽样负样本
for user in all_users:
    # 获取用户未打分的物品
    rated_items = interaction_matrix.loc[user][interaction_matrix.loc[user] > 0].index
    unrated_items = list(set(all_items) - set(rated_items))
    
    # 如果未打分物品为空，跳过该用户（其实对于互联网平台是不可能的，只是为了严谨性增加了这个步骤）
    if len(unrated_items) == 0:
        continue
    
    # 获取未打分物品的权重
    unrated_weights = item_popularity[item_popularity['movie_id'].isin(unrated_items)]['weight'].values
    
    # 归一化权重
    unrated_weights_normalized = unrated_weights / unrated_weights.sum()

    # 确定抽样数量：不能超过未打分物品的数量
    sample_size = min(len(rated_items), len(unrated_items))
    
    # 从未打分物品中按权重抽样，使用np.random.choice可抽样，p参数输入权重/概率值
    sampled_items = np.random.choice(unrated_items, size=sample_size, p=unrated_weights_normalized, replace=False)
    
    # 添加到负样本列表
    for item in sampled_items:
        negative_samples.append({'user_id': user, 'movie_id': item, 'rating': 0})

# 将负样本转换为DataFrame
negative_samples = pd.DataFrame(negative_samples)

In [46]:
unrated_weights

array([136.23494272, 102.22824908,  47.08002568, ...,  19.92027455,
        15.90541458,  87.42261523], shape=(3365,))

In [19]:
negative_samples

,user_id,movie_id,rating
0,1,3039,0
1,1,541,0
2,1,35,0
3,1,692,0
4,1,2316,0
...,...,...,...
999282,6040,1241,0
999283,6040,342,0
999284,6040,3053,0
999285,6040,1603,0


In [20]:
# 合并正负样本
ratings_sampled_data = pd.concat([positive_samples, negative_samples], ignore_index=True)

In [21]:
ratings_sampled_data

,user_id,movie_id,rating
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5
...,...,...,...
1999491,6040,1241,0
1999492,6040,342,0
1999493,6040,3053,0
1999494,6040,1603,0


In [22]:
users

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [23]:
merged_data = pd.merge(ratings_sampled_data, users, on='user_id')
merged_data = pd.merge(merged_data, movies, on='movie_id')

In [24]:
merged_data

,user_id,movie_id,rating,gender,age,occupation,zip,title,genres
0,1,1193,5,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...
1999491,6040,1241,0,M,25,6,11106,Braindead (1992),Comedy|Horror
1999492,6040,342,0,M,25,6,11106,Muriel's Wedding (1994),Comedy|Romance
1999493,6040,3053,0,M,25,6,11106,"Messenger: The Story of Joan of Arc, The (1999)",Drama|War
1999494,6040,1603,0,M,25,6,11106,Mimic (1997),Sci-Fi|Thriller


In [25]:
merged_data['label'] = (merged_data['rating'] >= 1).astype(int)

In [26]:
merged_data

,user_id,movie_id,rating,gender,age,occupation,zip,title,genres,label
0,1,1193,5,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,1
1,1,661,3,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,1
2,1,914,3,F,1,10,48067,My Fair Lady (1964),Musical|Romance,1
3,1,3408,4,F,1,10,48067,Erin Brockovich (2000),Drama,1
4,1,2355,5,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,1
...,...,...,...,...,...,...,...,...,...,...
1999491,6040,1241,0,M,25,6,11106,Braindead (1992),Comedy|Horror,0
1999492,6040,342,0,M,25,6,11106,Muriel's Wedding (1994),Comedy|Romance,0
1999493,6040,3053,0,M,25,6,11106,"Messenger: The Story of Joan of Arc, The (1999)",Drama|War,0
1999494,6040,1603,0,M,25,6,11106,Mimic (1997),Sci-Fi|Thriller,0


In [27]:
# 将用户和电影ID映射到连续的整数，与矩阵分解的操作一样
user_ids = merged_data['user_id'].unique()
movie_ids = merged_data['movie_id'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

merged_data['user_idx'] = merged_data['user_id'].map(user_to_idx)
merged_data['movie_idx'] = merged_data['movie_id'].map(movie_to_idx)
merged_data

,user_id,movie_id,rating,gender,age,occupation,zip,title,genres,label,user_idx,movie_idx
0,1,1193,5,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,1,0,0
1,1,661,3,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,1,0,1
2,1,914,3,F,1,10,48067,My Fair Lady (1964),Musical|Romance,1,0,2
3,1,3408,4,F,1,10,48067,Erin Brockovich (2000),Drama,1,0,3
4,1,2355,5,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,1,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...
1999491,6040,1241,0,M,25,6,11106,Braindead (1992),Comedy|Horror,0,6039,2052
1999492,6040,342,0,M,25,6,11106,Muriel's Wedding (1994),Comedy|Romance,0,6039,803
1999493,6040,3053,0,M,25,6,11106,"Messenger: The Story of Joan of Arc, The (1999)",Drama|War,0,6039,1168
1999494,6040,1603,0,M,25,6,11106,Mimic (1997),Sci-Fi|Thriller,0,6039,1664


In [28]:
# 将性别编码为数值
gender_encoder = LabelEncoder()
merged_data['gender_encoded'] = gender_encoder.fit_transform(merged_data['gender'])
merged_data['gender_encoded']

0          0
1          0
2          0
3          0
4          0
          ..
1999491    1
1999492    1
1999493    1
1999494    1
1999495    1
Name: gender_encoded, Length: 1999496, dtype: int64

In [29]:
merged_data

,user_id,movie_id,rating,gender,age,occupation,zip,title,genres,label,user_idx,movie_idx,gender_encoded
0,1,1193,5,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,1,0,0,0
1,1,661,3,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,1,0,1,0
2,1,914,3,F,1,10,48067,My Fair Lady (1964),Musical|Romance,1,0,2,0
3,1,3408,4,F,1,10,48067,Erin Brockovich (2000),Drama,1,0,3,0
4,1,2355,5,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,1,0,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999491,6040,1241,0,M,25,6,11106,Braindead (1992),Comedy|Horror,0,6039,2052,1
1999492,6040,342,0,M,25,6,11106,Muriel's Wedding (1994),Comedy|Romance,0,6039,803,1
1999493,6040,3053,0,M,25,6,11106,"Messenger: The Story of Joan of Arc, The (1999)",Drama|War,0,6039,1168,1
1999494,6040,1603,0,M,25,6,11106,Mimic (1997),Sci-Fi|Thriller,0,6039,1664,1


In [30]:
# 将电影类型编码进行multi-hot编码
genres = merged_data['genres'].str.get_dummies(sep='|')

In [31]:
genres

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999491,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0
1999492,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
1999493,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
1999494,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0


In [32]:
# 提取特征和标签
X_user = merged_data[['user_idx', 'gender_encoded', 'age', 'occupation']].values
X_movie = merged_data[['movie_idx']].values
X_genres = genres.values
y = merged_data['label'].values

用户属性，分别是用户ID，用户性别，用户年龄，用户职业

In [33]:
print(X_user.shape)
X_user

(1999496, 4)


array([[   0,    0,    1,   10],
       [   0,    0,    1,   10],
       [   0,    0,    1,   10],
       ...,
       [6039,    1,   25,    6],
       [6039,    1,   25,    6],
       [6039,    1,   25,    6]], shape=(1999496, 4))

电影的ID属性

In [34]:
print(X_movie.shape)
X_movie

(1999496, 1)


array([[   0],
       [   1],
       [   2],
       ...,
       [1168],
       [1664],
       [1802]], shape=(1999496, 1))

电影的分类属性，这里直接用了one-hot编码

In [35]:
print(X_genres.shape)
X_genres

(1999496, 18)


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1999496, 18))

是正样本还是负样本

In [36]:
print(y.shape)
y

(1999496,)


array([1, 1, 1, ..., 0, 0, 0], shape=(1999496,))

In [37]:
# 将数据转换为PyTorch张量
X_user = torch.tensor(X_user, dtype=torch.float32)
X_movie = torch.tensor(X_movie, dtype=torch.long)
X_genres = torch.tensor(X_genres, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

print(X_user.shape)
print(X_user.shape)

# 创建数据集和数据加载器
dataset = TensorDataset(X_user, X_movie, X_genres, y)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

torch.Size([1999496, 4])
torch.Size([1999496, 4])


### 双塔模型建模与训练

In [38]:
import torch
import torch.nn as nn

class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_movies, embedding_dim, num_genres):
        super(TwoTowerModel, self).__init__()
        # 用户塔
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.user_fc = nn.Sequential(
            nn.Linear(embedding_dim + 3, 64),  # 用户ID + 性别 + 年龄 + 职业
            nn.ReLU(),
            nn.Linear(64, 32)
        )
        
        # 电影塔
        self.movie_embedding = nn.Embedding(num_movies, embedding_dim)
        self.movie_fc = nn.Sequential(
            nn.Linear(embedding_dim + num_genres, 64),  # 电影ID + 电影类型
            nn.ReLU(),
            nn.Linear(64, 32)
        )
    
    def forward(self, X_user, X_movie, X_genres):
        # 用户嵌入
        user_idx = X_user[:, 0].long()
        user_embed = self.user_embedding(user_idx)
        user_embed = torch.cat([user_embed, X_user[:, 1:]], dim=1)
        user_embed = self.user_fc(user_embed)  # 形状: [batch_size, 32]
        
        # 电影嵌入
        movie_embed = self.movie_embedding(X_movie.squeeze())  # 形状: [batch_size, embedding_dim]
        movie_embed = torch.cat([movie_embed, X_genres], dim=1)  # 形状: [batch_size, embedding_dim + num_genres]
        movie_embed = self.movie_fc(movie_embed)  # 形状: [batch_size, 32]
        
        # 对用户嵌入和电影嵌入进行 L2 归一化
        user_embed_normalized = torch.nn.functional.normalize(user_embed, p=2, dim=1)
        movie_embed_normalized = torch.nn.functional.normalize(movie_embed, p=2, dim=1)
        
        # 计算余弦相似度
        cosine_similarity = (user_embed_normalized * movie_embed_normalized).sum(dim=1, keepdim=True)  # 形状: [batch_size, 1]
        
        # 应用 Sigmoid 函数，输出二分类结果
        output = torch.sigmoid(cosine_similarity)  # 形状: [batch_size, 1]
        return output.squeeze()  # 形状: [batch_size]

In [39]:
# 初始化模型、损失函数和优化器
num_users = len(user_ids)
num_movies = len(movie_ids)
embedding_dim = 8
num_genres = genres.shape[1]

model = TwoTowerModel(num_users, num_movies, embedding_dim, num_genres)
# criterion = nn.BCEWithLogitsLoss()  # 适用于二分类任务
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型，这里为了测试选了3轮，可适当调大维度
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_user, X_movie, X_genres, y in dataloader:
        optimizer.zero_grad()
        outputs = model(X_user, X_movie, X_genres)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [1/3], Loss: 0.6622
Epoch [2/3], Loss: 0.6365
Epoch [3/3], Loss: 0.6267


这里用了交叉熵作损失函数，pytorch的交叉熵损失函数是通过nn.BCELoss()来调用的，对应TensorFlow的BinaryCrossEntropy</br>
交叉熵对于随机猜测的损失值为：−ln(0.5)≈0.693

### 为1号用户作推荐

In [40]:
# 获取1号用户的编码索引（假设user_id=1对应user_idx=0）
user_idx = 0  # 需确认merged_data中user_id=1对应的user_idx值
user_features = merged_data[merged_data['user_idx'] == user_idx].iloc[0]
X_user_sample = torch.tensor([[user_idx, 
                              user_features['gender_encoded'],
                              user_features['age'],
                              user_features['occupation']]], 
                              dtype=torch.float32)

X_user_sample

tensor([[ 0.,  0.,  1., 10.]])

In [41]:
# 获取用户已观看的电影（避免重复推荐）
watched_movies = merged_data[merged_data['user_idx'] == user_idx]['movie_idx'].tolist()

# 生成所有未观看的电影候选集
all_movie_indices = merged_data['movie_idx'].unique()
candidate_movies = [m for m in all_movie_indices if m not in watched_movies]

# 候选电影数
num_candidates = len(candidate_movies)

# 用户特征扩展（确保与候选电影数量一致）
X_user_repeated = X_user_sample.repeat(num_candidates, 1)  # 形状 [num_candidates, 4]

# 电影特征构造（必须保持维度一致性）
X_movie_candidates = torch.tensor(candidate_movies, dtype=torch.long).unsqueeze(1)  # 形状 [num_candidates, 1]
X_genres_candidates = torch.tensor(genres.loc[candidate_movies].values, dtype=torch.float32)  # 形状 [num_candidates, 18]

print(X_movie_candidates)
print(X_genres_candidates)

tensor([[  53],
        [  54],
        [  55],
        ...,
        [3703],
        [3704],
        [3705]])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [42]:
model.eval()
with torch.no_grad():
    # 重复用户特征以匹配候选电影数量
    X_user_repeated = X_user_sample.repeat(len(candidate_movies), 1)
    
    # 预测
    predictions = model(X_user_repeated, X_movie_candidates.unsqueeze(1), X_genres_candidates)
    probabilities = predictions.numpy()

In [43]:
probabilities

array([0.5044165 , 0.39147457, 0.44311813, ..., 0.46980044, 0.3858506 ,
       0.2872241 ], shape=(3600,), dtype=float32)

In [44]:
# 将概率与电影ID关联
recommend_df = pd.DataFrame({
    'movie_idx': candidate_movies,
    'probability': probabilities
})

# 获取电影标题并排序
recommend_df = recommend_df.merge(merged_data[['movie_idx', 'title']].drop_duplicates(), on='movie_idx')
top_n_recommendations = recommend_df.sort_values('probability', ascending=False).head(10)

打印推荐列表

In [45]:
top_n_recommendations

,movie_idx,probability,title
399,464,0.727935,William Shakespeare's Romeo and Juliet (1996)
816,898,0.726931,Stir of Echoes (1999)
3352,3458,0.726049,Black Sunday (La Maschera Del Demonio) (1960)
1860,1961,0.725957,"Hunger, The (1983)"
417,482,0.724262,Interview with the Vampire (1994)
1194,1284,0.724214,"Killer, The (Die xue shuang xiong) (1989)"
3493,3599,0.724084,Mutters Courage (1995)
1541,1636,0.723632,American Pop (1981)
1008,1095,0.722343,Howard the Duck (1986)
768,850,0.719992,Rising Sun (1993)
